# Optimized KMeans Clustering — Physical Activity Dataset
**Pipeline:** Load → Encode → IQR Outlier Removal → Scale → PCA Check → K Selection → Final Model → Profile → Visualize → Export

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score,
                              davies_bouldin_score,
                              calinski_harabasz_score)
import plotly.express as px

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('All imports OK')

## 1. Load & Inspect

In [ ]:
df_raw = pd.read_csv('physical_activity.csv')

print('=== Shape ===')
print(df_raw.shape)

print('\n=== Data Types ===')
print(df_raw.dtypes)

print('\n=== Missing Values ===')
print(df_raw.isnull().sum())

print('\n=== Value Ranges (numeric) ===')
print(df_raw.describe())

print('\n=== Sample Rows ===')
df_raw.head()

## 2. Feature Selection & Encoding

**Strategy for this dataset:**
- Drop: `Unnamed: 0` (row index), `Description` (free text — not suitable for KMeans), `Cluster` (pre-existing label — would leak the answer), `SVD_1`, `SVD_2` (exact duplicates of `x` and `y`)
- Keep numeric: `x`, `y`, `z`
- Encode with `LabelEncoder`: `Actvitiy` (categorical nominal — encode to ordinal integer)
- No binary yes/no columns exist in this dataset

In [ ]:
df = df_raw.copy()

# ── Drop columns not suitable for clustering ──────────────────────────────────
drop_cols = ['Unnamed: 0', 'Description', 'Cluster', 'SVD_1', 'SVD_2']
df.drop(columns=drop_cols, inplace=True)
print('After dropping metadata/duplicate columns:', df.columns.tolist())

# ── Detect column types ───────────────────────────────────────────────────────
categorical_cols = df.select_dtypes(include='object').columns.tolist()
numeric_cols     = df.select_dtypes(include=[np.number]).columns.tolist()

# ── Detect binary yes/no columns (values only in {0,1} or {'yes','no'}) ──────
binary_cols = []
for col in numeric_cols:
    unique_vals = set(df[col].dropna().unique())
    if unique_vals <= {0, 1}:
        binary_cols.append(col)
for col in categorical_cols:
    unique_vals = {str(v).strip().lower() for v in df[col].dropna().unique()}
    if unique_vals <= {'yes', 'no'}:
        binary_cols.append(col)

non_binary_numeric = [c for c in numeric_cols if c not in binary_cols]

print(f'\nCategorical columns  : {categorical_cols}')
print(f'Numeric columns      : {numeric_cols}')
print(f'Binary yes/no columns: {binary_cols}')
print(f'Non-binary numeric   : {non_binary_numeric}')

# ── LabelEncode categorical (and any binary string) columns ──────────────────
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f'  LabelEncoded "{col}" → {le.classes_[:5]} ...')

# Ordinal integer columns are left as-is
print('\nDataFrame after encoding:')
print(df.dtypes)
print(df.head())

## 3. IQR Outlier Removal (non-binary numeric columns only)

In [ ]:
df_pre_iqr = df.copy()
print(f'Rows before IQR removal: {len(df_pre_iqr)}')

# Recompute non_binary_numeric after encoding (all cols are now numeric)
all_numeric_now = df.select_dtypes(include=[np.number]).columns.tolist()
# Skip encoded binary cols (if any) from IQR
iqr_cols = [c for c in all_numeric_now if c not in binary_cols]
print(f'Columns used for IQR: {iqr_cols}')

mask = pd.Series(True, index=df.index)
for col in iqr_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    col_mask = (df[col] >= lower) & (df[col] <= upper)
    removed  = (~col_mask).sum()
    print(f'  {col:20s}: [{lower:.4f}, {upper:.4f}]  removed {removed} rows')
    mask &= col_mask

df_clean = df[mask].copy().reset_index(drop=True)
print(f'\nRows after IQR removal : {len(df_clean)}')
print(f'Rows removed           : {len(df_pre_iqr) - len(df_clean)}')

## 4. Drop Constant Columns (std = 0)

In [ ]:
stds = df_clean.std(numeric_only=True)
constant_cols = list(np.where(stds == 0)[0])   # indices
constant_col_names = stds.index[np.where(stds == 0)[0]].tolist()

if constant_col_names:
    print(f'Dropping constant columns (std=0): {constant_col_names}')
    df_clean.drop(columns=constant_col_names, inplace=True)
else:
    print('No constant columns found — all features retained.')

print('Features for clustering:', df_clean.columns.tolist())

## 5. StandardScaler

In [ ]:
feature_cols = df_clean.columns.tolist()
X_raw = df_clean[feature_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f'X_scaled shape : {X_scaled.shape}')
print(f'Mean (should ≈ 0): {X_scaled.mean(axis=0).round(4)}')
print(f'Std  (should ≈ 1): {X_scaled.std(axis=0).round(4)}')

## 6. PCA Check — Does 90% variance need all components?

In [ ]:
n_features = X_scaled.shape[1]
pca_check  = PCA(n_components=n_features, random_state=RANDOM_STATE)
pca_check.fit(X_scaled)

cumvar = np.cumsum(pca_check.explained_variance_ratio_)
n_for_90 = int(np.argmax(cumvar >= 0.90)) + 1

print(f'Total features      : {n_features}')
print(f'Components for 90%  : {n_for_90}')
print(f'Cumulative variance : {cumvar.round(4)}')

plt.figure(figsize=(7, 4))
plt.plot(range(1, n_features + 1), cumvar, marker='o', color='steelblue')
plt.axhline(0.90, color='red', linestyle='--', label='90% threshold')
plt.axvline(n_for_90, color='orange', linestyle=':', label=f'n={n_for_90}')
plt.title('Cumulative Explained Variance by PCA')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Variance')
plt.legend()
plt.tight_layout()
plt.show()

if n_for_90 >= n_features:
    USE_PCA = False
    X_final = X_scaled
    print(f'\n⚠  90% variance requires ALL {n_features} components → PCA SKIPPED for clustering.')
    print(f'   X_final shape: {X_final.shape}')
else:
    USE_PCA = True
    pca_final = PCA(n_components=n_for_90, random_state=RANDOM_STATE)
    X_final   = pca_final.fit_transform(X_scaled)
    print(f'\n✔  PCA applied: {n_features} → {n_for_90} components.')
    print(f'   X_final shape: {X_final.shape}')

## 7. K Selection Loop (K = 2 to 12)

In [ ]:
SAMPLE_SIZE = min(2000, len(X_final))   # 2,000-point sample for slow metrics
K_RANGE     = range(2, 13)

wcss_scores = []
sil_scores  = []
db_scores   = []
ch_scores   = []

rng = np.random.RandomState(RANDOM_STATE)

print(f'Evaluating K = 2 to 12  |  sample_size={SAMPLE_SIZE}  |  n_init=10')
print('-' * 70, flush=True)

for k in K_RANGE:
    # ── Fit on full data ─────────────────────────────────────────────────────
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels_full = km.fit_predict(X_final)
    wcss = km.inertia_

    # ── Sample 2,000 points for slow metrics ────────────────────────────────
    idx_sample = rng.choice(len(X_final), size=SAMPLE_SIZE, replace=False)
    X_samp     = X_final[idx_sample]
    L_samp     = labels_full[idx_sample]

    sil = silhouette_score(X_samp, L_samp, random_state=RANDOM_STATE)
    db  = davies_bouldin_score(X_samp, L_samp)
    ch  = calinski_harabasz_score(X_samp, L_samp)

    wcss_scores.append(wcss)
    sil_scores.append(sil)
    db_scores.append(db)
    ch_scores.append(ch)

    print(f'K={k:2d}  WCSS={wcss:10.2f}  Silhouette={sil:.4f}  '
          f'Davies-Bouldin={db:.4f}  Calinski-Harabasz={ch:.2f}', flush=True)

print('-' * 70, flush=True)
print('Loop complete.', flush=True)

## 8. Plot All 4 Metrics (2×2 Grid)

In [ ]:
K_list = list(K_RANGE)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('KMeans Cluster Selection Metrics', fontsize=16, fontweight='bold')

# WCSS (Elbow)
ax = axes[0, 0]
ax.plot(K_list, wcss_scores, marker='o', color='steelblue', linewidth=2)
ax.set_title('WCSS (Elbow Curve)')
ax.set_xlabel('K'); ax.set_ylabel('WCSS')
ax.grid(True, alpha=0.3)

# Silhouette — higher is better
ax = axes[0, 1]
ax.plot(K_list, sil_scores, marker='s', color='green', linewidth=2)
ax.set_title('Silhouette Score  (higher = better)')
ax.set_xlabel('K'); ax.set_ylabel('Silhouette')
ax.grid(True, alpha=0.3)

# Davies-Bouldin — lower is better
ax = axes[1, 0]
ax.plot(K_list, db_scores, marker='^', color='tomato', linewidth=2)
ax.set_title('Davies-Bouldin Score  (lower = better)')
ax.set_xlabel('K'); ax.set_ylabel('Davies-Bouldin')
ax.grid(True, alpha=0.3)

# Calinski-Harabasz — higher is better
ax = axes[1, 1]
ax.plot(K_list, ch_scores, marker='D', color='darkorchid', linewidth=2)
ax.set_title('Calinski-Harabasz Score  (higher = better)')
ax.set_xlabel('K'); ax.set_ylabel('Calinski-Harabasz')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cluster_metrics_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved cluster_metrics_grid.png')

## 9. Pick Best K
Rule: choose K where **Silhouette AND Calinski-Harabasz both peak** — skip K=2 even if it scores best.

In [ ]:
k_vals   = np.array(K_list)
sil_arr  = np.array(sil_scores)
ch_arr   = np.array(ch_scores)

# Exclude K=2 (index 0)
valid_mask = k_vals > 2
k_valid    = k_vals[valid_mask]
sil_valid  = sil_arr[valid_mask]
ch_valid   = ch_arr[valid_mask]

# Normalise both metrics to [0,1] then sum for a combined score
sil_norm = (sil_valid - sil_valid.min()) / (sil_valid.max() - sil_valid.min() + 1e-9)
ch_norm  = (ch_valid  - ch_valid.min())  / (ch_valid.max()  - ch_valid.min()  + 1e-9)
combined = sil_norm + ch_norm

best_idx = int(np.argmax(combined))
BEST_K   = int(k_valid[best_idx])

print('K  | Silhouette | CH Score  | Combined')
print('-' * 45)
for i, k in enumerate(k_valid):
    marker = ' ◀ BEST' if k == BEST_K else ''
    print(f'K={k:2d} | {sil_valid[i]:.4f}     | {ch_valid[i]:9.2f} | {combined[i]:.4f}{marker}')

print(f'\n→ Best K selected: {BEST_K}')

## 10. Fit Final Model (n_init=50)

In [ ]:
final_model = KMeans(n_clusters=BEST_K, random_state=RANDOM_STATE, n_init=50)
final_labels = final_model.fit_predict(X_final)

df_clean['Cluster'] = final_labels

print(f'Final model fitted with K={BEST_K}, n_init=50')
print(f'Inertia : {final_model.inertia_:.4f}')
print(f'Cluster distribution:')
print(df_clean['Cluster'].value_counts().sort_index())

## 11. Cluster Profiling & Naming

In [ ]:
# Add back original Activity column for richer profiling
df_clean['Activity_raw'] = df_raw.loc[df_raw.index.isin(df_clean.index) 
                                      if False else slice(None), 'Actvitiy'].values[:len(df_clean)]

profile = df_clean.groupby('Cluster').mean(numeric_only=True).round(4)

print('=== Cluster Profiles (mean of scaled & encoded features) ===')
print(profile.to_string())

# ── Per-cluster top Activity (mode) ──────────────────────────────────────────
print('\n=== Dominant Activity per Cluster ===')
top_activity = df_clean.groupby('Cluster')['Activity_raw'].agg(
    lambda x: x.value_counts().index[0]
)
print(top_activity)

# ── Auto-name clusters based on standout features ────────────────────────────
# Compare each cluster's profile vs. global mean; largest positive deviation = standout
global_mean = profile.mean()
diff = profile.subtract(global_mean)

# Map feature-name hints
feature_hint = {
    'x'         : 'High-X motion',
    'y'         : 'High-Y motion',
    'z'         : 'High-Z motion',
    'Actvitiy'  : 'Activity-driven',
}

cluster_names = {}
for c in profile.index:
    top_feat = diff.loc[c].abs().idxmax()
    direction = 'High' if diff.loc[c, top_feat] > 0 else 'Low'
    dom_act   = top_activity[c]
    name = f'Cluster {c}: {direction}-{top_feat} / {dom_act}'
    cluster_names[c] = name
    print(f'  {name}')

df_clean['Cluster_Name'] = df_clean['Cluster'].map(cluster_names)

## 12. Cluster Profile Heatmap

In [ ]:
# Use only truly numeric feature columns for heatmap (drop Cluster_Name, Activity_raw)
heatmap_cols = [c for c in profile.columns if c not in ['Cluster_Name', 'Activity_raw']]
heatmap_data = profile[heatmap_cols]

plt.figure(figsize=(max(8, len(heatmap_cols) * 1.5), max(5, BEST_K * 0.8)))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.3f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    yticklabels=[cluster_names[c] for c in heatmap_data.index]
)
plt.title(f'Cluster Profile Heatmap  (K={BEST_K})', fontsize=14, fontweight='bold')
plt.xlabel('Feature')
plt.ylabel('Cluster')
plt.tight_layout()
plt.savefig('cluster_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved cluster_heatmap.png')

## 13. 2D Scatter using PCA (visualization only)

In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d   = pca_2d.fit_transform(X_scaled)   # always from scaled, never from clustering PCA

var_exp = pca_2d.explained_variance_ratio_
print(f'2D PCA variance explained: PC1={var_exp[0]:.3f}, PC2={var_exp[1]:.3f}  '
      f'(total={sum(var_exp):.3f})')

palette = sns.color_palette('tab10', n_colors=BEST_K)

plt.figure(figsize=(10, 7))
for c in sorted(df_clean['Cluster'].unique()):
    mask_c = df_clean['Cluster'].values == c
    plt.scatter(
        X_2d[mask_c, 0], X_2d[mask_c, 1],
        label=cluster_names[c],
        color=palette[c],
        alpha=0.6, s=40, edgecolors='none'
    )

plt.title(f'2D PCA Cluster Scatter  (K={BEST_K})', fontsize=13, fontweight='bold')
plt.xlabel(f'PC1 ({var_exp[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({var_exp[1]*100:.1f}%)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('cluster_scatter_2d.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved cluster_scatter_2d.png')

## 14. 3D Scatter using Plotly → saved as HTML
> **Note:** Saved as `.html` file — open in any browser. `fig.show()` is skipped because WebGL is not supported in your Jupyter/browser environment.

In [ ]:
pca_3d = PCA(n_components=3, random_state=RANDOM_STATE)
X_3d   = pca_3d.fit_transform(X_scaled)

df_3d = pd.DataFrame({
    'PC1'          : X_3d[:, 0],
    'PC2'          : X_3d[:, 1],
    'PC3'          : X_3d[:, 2],
    'Cluster'      : df_clean['Cluster'].astype(str),
    'Cluster_Name' : df_clean['Cluster_Name'],
    'Activity'     : df_clean['Activity_raw'],
})

var3 = pca_3d.explained_variance_ratio_

fig = px.scatter_3d(
    df_3d,
    x='PC1', y='PC2', z='PC3',
    color='Cluster',
    hover_name='Activity',
    hover_data={'Cluster_Name': True, 'PC1': ':.3f', 'PC2': ':.3f', 'PC3': ':.3f'},
    title=f'3D PCA Cluster Scatter  (K={BEST_K})  '
          f'[PC1={var3[0]*100:.1f}% | PC2={var3[1]*100:.1f}% | PC3={var3[2]*100:.1f}%]',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    template='plotly_white',
    opacity=0.75
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(
    scene=dict(
        xaxis_title=f'PC1 ({var3[0]*100:.1f}%)',
        yaxis_title=f'PC2 ({var3[1]*100:.1f}%)',
        zaxis_title=f'PC3 ({var3[2]*100:.1f}%)'
    ),
    legend_title='Cluster',
    margin=dict(l=0, r=0, b=0, t=60)
)

# ── Save to HTML (do NOT call fig.show() — WebGL not supported) ─────────────
html_path = 'cluster_3d_scatter.html'
fig.write_html(html_path, include_plotlyjs='cdn')
print(f'3D scatter saved to: {html_path}')
print('Open this file in Chrome/Edge/Firefox to explore the interactive 3D plot.')

## 15. Export: Clustered CSV + Model PKL + Scaler PKL

In [ ]:
# ── 1. Clustered CSV ──────────────────────────────────────────────────────────
csv_out = 'physical_activity_clustered.csv'
df_clean.to_csv(csv_out, index=False)
print(f'Clustered CSV saved  → {csv_out}  ({len(df_clean)} rows)')

# ── 2. KMeans model PKL ───────────────────────────────────────────────────────
model_out = 'kmeans_model.pkl'
with open(model_out, 'wb') as f:
    pickle.dump(final_model, f)
print(f'KMeans model saved   → {model_out}')

# ── 3. Scaler PKL ────────────────────────────────────────────────────────────
scaler_out = 'scaler.pkl'
with open(scaler_out, 'wb') as f:
    pickle.dump(scaler, f)
print(f'StandardScaler saved → {scaler_out}')

# ── 4. Label encoders PKL (bonus — needed to inverse-transform predictions) ──
le_out = 'label_encoders.pkl'
with open(le_out, 'wb') as f:
    pickle.dump(label_encoders, f)
print(f'LabelEncoders saved  → {le_out}')

print('\n=== All outputs saved ===')
print(f'  {csv_out}')
print(f'  {model_out}')
print(f'  {scaler_out}')
print(f'  {le_out}')
print(f'  cluster_metrics_grid.png')
print(f'  cluster_heatmap.png')
print(f'  cluster_scatter_2d.png')
print(f'  {html_path}')

---
## Quick Reload & Predict (re-use saved artefacts)
Run this cell anytime to load the trained model and predict on new data.

In [ ]:
# ── Load saved artefacts ─────────────────────────────────────────────────────
with open('kmeans_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
with open('scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)
with open('label_encoders.pkl', 'rb') as f:
    loaded_le = pickle.load(f)

print('Model  :', loaded_model)
print('Scaler :', loaded_scaler)
print('LEs    :', list(loaded_le.keys()))

# ── Example: predict cluster for new rows ────────────────────────────────────
# new_data = pd.DataFrame({...})   # must have same feature columns
# new_scaled = loaded_scaler.transform(new_data)
# new_labels = loaded_model.predict(new_scaled)
# print(new_labels)